# Empirical risk curves on the ACE2 deep mutational scan

Computes the experimental test risk of ridge regression on the SARS-CoV-2
ACE2-binding DMS data (Starr et al. 2020, *Deep mutational scanning of SARS-CoV-2
receptor binding domain*) on a grid of aspect ratios $r = qL/M$ and ridge
penalties $\lambda$, and saves the result to CSV.

**Output columns:** `Ab, q, L, M, r, mu, R, sR, nTest, nTrial` — `R` is the test
mean squared error averaged over `nTrial` random train/test realizations and
`sR` its **standard deviation over trials** (used downstream for error bars).

The companion notebook `risk_curves_fit.ipynb` fits the theoretical curves to
this CSV.

Requirements: `numpy`, `pandas`, `scipy`, `scikit-learn`, `joblib`,
`threadpoolctl`, `tqdm`.

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.linear_model import Ridge
from joblib import Parallel, delayed
from threadpoolctl import threadpool_limits
from tqdm.auto import tqdm

# ---------------------------------------------------------------------------
# Paths and reference sequence
# ---------------------------------------------------------------------------
RAW_DATA_PATH = Path('../data/ACE2_Wuhan_raw_data.csv')      # raw ACE2 DMS measurements (Starr et al. 2020)
OUTPUT_CSV = Path('../data/ACE2_Wuhan_risk_landscape.csv')

# Wuhan-Hu-1 reference RBD: L = 201 sites (spike residues 331-531)
WT = ("NITNLCPFGEVFNATRFASVYAWNRKRISNCVADYSVLYNSASFSTFKCYGVSPTKLNDLCFTNVYADSFVI"
      "RGDEVRQIAPGQTGKIADYNYKLPDDFTGCVIAWNSNNLDSKVGGNYNYLYRLFRKSNLKPFERDISTEIY"
      "QAGSTPCNGVEGFNCYFPLQSYGFQPTNGVGYQPYRVVVLSFELLHAPATVCGPKKST")

AA = list('ACDEFGHIKLMNPQRSTVWY')          # q = 20 amino acids
aa_id = {a: i for i, a in enumerate(AA)}

q = len(AA)
L = len(WT)
p = q * L                                  # one-hot feature dimension
WT_SEQ = np.array([aa_id[c] for c in WT], dtype=np.int32)

/home/alessandro/miniconda3/envs/escape_map_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Data preprocessing

Keep variants with a measured binding score $\Delta\log_{10} K_a$ (relative to
the wild type) and remove variants containing stop codons. After this step
146,437 variants remain for the ACE2 dataset.

In [2]:
df = pd.read_csv(RAW_DATA_PATH)
df = df[df['delta_log10Ka'].notna()].copy()
df = df[~df['aa_substitutions'].str.contains(r'\*', na=False)].copy()
df = df.reset_index(drop=True)

n_samples = len(df)
print(f'{n_samples:,} variants after preprocessing')

# empirical mutation rate: f = (mean number of mutations per variant) / L
lambda_hat = df['n_aa_substitutions'].mean()
f = lambda_hat / L
print(f'mean mutations per variant = {lambda_hat:.4f}   ->   f = {f:.6f}')

146,437 variants after preprocessing
mean mutations per variant = 2.5833   ->   f = 0.012852


In [ ]:
_MUT_RE = re.compile(r'^([A-Z])(\d+)([A-Z])$')

def apply_aa_substitutions(wt_seq, aa_substitutions):
    """Return the integer-encoded sequence carrying the given substitutions.

    `aa_substitutions` is a space-separated string such as "A22C R127G",
    with 1-based positions on the RBD region. The wild type is returned
    unchanged for variants without substitutions.
    """
    seq = wt_seq.copy()
    if pd.isna(aa_substitutions) or str(aa_substitutions).strip() == '':
        return seq
    for mut in str(aa_substitutions).split():
        m = _MUT_RE.match(mut)
        if not m:
            raise ValueError(f'Bad mutation format: {mut!r}')
        old_aa, pos, new_aa = m.group(1), int(m.group(2)), m.group(3)
        idx = pos - 1
        if idx < 0 or idx >= len(seq):
            raise IndexError(f'Position {pos} out of range for length {len(seq)}')
        if wt_seq[idx] != aa_id[old_aa]:
            print(f'WT residue mismatch at position {pos}: '
                  f'expected {aa_id[old_aa]}, found {aa_id[new_aa]}')
        seq[idx] = aa_id[new_aa]
    return seq

## Settings

- Grid: 15 aspect ratios geometrically spaced in $r \in [0.03, 10]$, realized
  through $M = \lfloor qL/r \rfloor$ training samples, and 7 ridge penalties
  $\lambda \in \{10^{-6}, \dots, 10^{0}\}$.
- Ridge convention: minimize $\sum_\mu (G_\mu - g\,\delta_\mu - g_0)^2 +
  M\lambda\,\|g\|^2$ with an intercept $g_0$ absorbing the additive constant
  (`alpha = M * lambda`, `fit_intercept=True`).
- For each of `n_trials = 5` trials, $M + M_{\text{test}}$ distinct variants are
  drawn uniformly without replacement (seed 12345); the first $M_{\text{test}}$
  form the test set.

In [ ]:
Mtest = 1000
n_trials = 5

r_values = np.geomspace(0.03, 10, 15)
mu_grid = 10.0 ** np.arange(-6, 1, dtype=float)

SEED = 12345
TOL = 1e-3        # LSQR convergence tolerance (sufficient for risk evaluation)
N_JOBS = 4        # simultaneous ridge fits over the lambda grid

## One-hot encoding

The full sparse design matrix is built once in CSR format: exactly one nonzero
entry per site, column index `site * q + amino_acid`.

In [ ]:
nnz = n_samples * L
indices = np.empty(nnz, dtype=np.int32)
indptr = np.arange(0, nnz + 1, L, dtype=np.int32)
site_offsets = np.arange(L, dtype=np.int32) * q

for i, substitutions in enumerate(tqdm(df['aa_substitutions'].to_numpy())):
    seq = apply_aa_substitutions(WT_SEQ, substitutions)
    indices[i * L:(i + 1) * L] = site_offsets + seq

X_all = csr_matrix((np.ones(nnz, dtype=np.float32), indices, indptr),
                   shape=(n_samples, p))
G_all = df['delta_log10Ka'].to_numpy(dtype=np.float32)
print(f'design matrix: {X_all.shape}, {X_all.nnz:,} nonzero entries')

## Ridge sweep

For each aspect ratio, trials are drawn once and shared across the lambda grid;
the penalties are fitted in parallel threads (the sparse design matrix is
shared, no large copies between workers).

In [ ]:
def fit_one_mu(X_train, y_train, X_test, y_test, mu, M):
    model = Ridge(alpha=M * mu, fit_intercept=True,
                  solver='lsqr', tol=TOL, max_iter=300, copy_X=False)
    model.fit(X_train, y_train)
    return np.mean((model.predict(X_test) - y_test) ** 2)


rng = np.random.default_rng(SEED)
results = []

for r_goal in r_values:
    M = int(p / r_goal)
    r = p / M
    if M + Mtest > n_samples:
        continue
    print(f'r = {r:.5g}, M = {M:,}')

    risk_values = np.empty((n_trials, len(mu_grid)))
    for trial in tqdm(range(n_trials), desc=f'M={M}'):
        idx = rng.choice(n_samples, size=M + Mtest, replace=False)
        test_idx, train_idx = idx[:Mtest], idx[Mtest:]
        X_train, X_test = X_all[train_idx], X_all[test_idx]
        y_train, y_test = G_all[train_idx], G_all[test_idx]

        with threadpool_limits(limits=1):
            out = Parallel(n_jobs=N_JOBS, prefer='threads')(
                delayed(fit_one_mu)(X_train, y_train, X_test, y_test, mu, M)
                for mu in mu_grid)
        risk_values[trial] = out

    for j, mu in enumerate(mu_grid):
        results.append({
            'Ab': 'ACE2', 'q': q, 'L': L, 'M': M, 'r': r, 'mu': mu,
            'R': risk_values[:, j].mean(),
            'sR': risk_values[:, j].std(),   # trial standard deviation
            'nTest': Mtest, 'nTrial': n_trials,
        })

results_df = pd.DataFrame(results)
results_df

In [ ]:
results_df.to_csv(OUTPUT_CSV, index=False)
print(f'Saved {OUTPUT_CSV.resolve()}')